# 第一阶段研究变量构造

本 Notebook 只读取 `research_panel_base`，构造第一阶段财务、公司特征和创新变量。缺失值依照变量字典传播，不做 winsorization。

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')
from src.build_research_variables import load_and_build_variables, variable_summary

processed_dir = Path('../data/processed')
variables = load_and_build_variables(processed_dir)
variables.head()

,stock_code,company_name,year,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,...,rd_intensity,cash_ratio,employee_ln,firm_age,patent_total,patent_total_ln,invention_ln,citation_ln,invention_share,citations_per_invention
0,000001,华辰科技股份有限公司,2020,44813001602.0,23908847993.0,8788852319.0,823332467.0,4322096458.0,655221236.0,0.0394,...,0.074551,0.096447,9.597302,16.0,27.0,3.332205,2.197225,3.526361,0.296296,4.125
1,000001,华辰科技股份有限公司,2021,51047456089.0,30247752226.0,12258996387.0,1406421903.0,12041496671.0,680417512.0,0.0676,...,0.055504,0.235888,9.825202,17.0,24.0,3.218876,2.197225,3.178054,0.333333,2.875
2,000001,华辰科技有限公司,2022,53781621074.0,40472674936.0,11646604275.0,-90319782.0,8762245930.0,883046862.0,-0.0068,...,0.075820,0.162923,9.733292,18.0,NaN,NaN,NaN,NaN,NaN,NaN
3,000001,华辰科技股份有限公司,2023,1934677743.0,476851735.0,17627268570.0,-908582885.0,273990284.0,393474856.0,-0.6232,...,0.022322,0.141621,7.977282,19.0,17.0,2.890372,2.197225,3.218876,0.470588,3.000
4,000001,ST华辰科技股份有限公司,2024,48924685296.0,37663675836.0,35182016317.0,2123243663.0,13679245844.0,2724202038.0,0.1885,...,0.077432,0.279598,9.661225,20.0,18.0,2.944439,2.397895,3.401197,0.555556,2.900


In [2]:
summary = variable_summary(variables)
summary

,variable,N,missing,min,median,max
0,size_ln,239,1,20.531485,24.140172,24.811137
1,leverage,238,2,0.156780,0.465852,0.798617
2,roa,237,3,-1.136998,0.028604,8.951074
3,rd_intensity,239,1,0.005026,0.059510,0.118347
4,cash_ratio,238,2,0.032171,0.173097,0.299151
5,employee_ln,238,2,5.505332,9.935185,10.713551
6,firm_age,180,60,2.000000,13.000000,21.000000
7,invention_patents,219,21,2.000000,7.000000,16.000000
8,utility_patents,219,21,4.000000,13.000000,25.000000
9,patent_citations,218,22,5.000000,22.000000,64.000000


In [3]:
assert variables.shape[0] == 240
assert not variables.duplicated(['stock_code', 'year']).any()
numeric = variables.select_dtypes(include='number')
assert not numeric.isin([float('inf'), float('-inf')]).any().any()
derived_patents = [
    'patent_total', 'patent_total_ln', 'invention_ln',
    'citation_ln', 'invention_share', 'citations_per_invention',
]
assert variables.loc[variables['patent_record_present'] == 0, derived_patents].isna().all().all()
variables.to_parquet(processed_dir / 'research_panel_variables.parquet', index=False)
stata = variables.copy()
stata_columns = [
    'year', 'invention_patents', 'utility_patents',
    'patent_record_present', 'size_ln', 'leverage', 'roa',
    'rd_intensity', 'cash_ratio', 'employee_ln', 'firm_age',
    'patent_total', 'patent_total_ln', 'invention_ln',
    'citation_ln', 'invention_share', 'citations_per_invention',
]
for column in stata_columns:
    stata[column] = stata[column].astype(float)
stata.to_stata(processed_dir / 'research_panel_variables.dta', write_index=False, version=118)
print('Research variables written successfully.')

Research variables written successfully.


## 变量边界

比例变量仅在分母严格为正时计算；log 变量仅在输入满足正值条件时计算；上市日期缺失不推测上市年份；专利未观测记录的衍生变量全部保持 missing。